# NLP Lab Assignment 8 — POS Tagging with BiLSTM and BERT

In [ ]:
# Cell 1 — Install dependencies
!pip -q install datasets transformers accelerate scikit-learn seaborn

In [ ]:
# Cell 2 — Imports, reproducibility, and device
import os, random, time, copy, collections, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PyTorch:", torch.__version__)

In [ ]:
# Cell 3 — Q1(a): Load Universal Dependencies English EWT official splits
# Hugging Face dataset configuration: en_ewt
raw = load_dataset("universal_dependencies", "en_ewt")
print(raw)
for split in ["train", "validation", "test"]:
    print(f"{split}: {len(raw[split]):,} sentences")

In [ ]:
# Cell 4 — Prepare word-level tokens and UPOS labels
# UD UPOS tags are stored in the 'upos' field. Remove empty/multiword-token rows
# by retaining only entries whose UPOS label is not None.
def extract_sentences(split):
    output = []
    for row in raw[split]:
        words, tags = [], []
        for word, tag in zip(row["tokens"], row["upos"]):
            if word is not None and tag is not None:
                words.append(str(word))
                tags.append(str(tag))
        if words:
            output.append((words, tags))
    return output

train_data = extract_sentences("train")
dev_data   = extract_sentences("validation")
test_data  = extract_sentences("test")

print("Usable sentence counts:")
print("Train:", len(train_data), "Dev:", len(dev_data), "Test:", len(test_data))
print("Example:", train_data[0])

In [ ]:
# Cell 5 — Q1(b): Show five word-level tagged sentences and ambiguous words
for i, (words, tags) in enumerate(train_data[:5], start=1):
    print(f"Sentence {i}:")
    print(" ".join(words))
    print(list(zip(words, tags)))
    print()

# Examples of context-dependent ambiguity to inspect in this corpus.
# The same surface form may receive different UPOS labels in different sentences.
target_words = {"book", "like", "work", "play", "watch"}
ambiguous_examples = []
for words, tags in train_data:
    for w, t in zip(words, tags):
        if w.lower() in target_words:
            ambiguous_examples.append((w, t, " ".join(words)))
    if len(ambiguous_examples) >= 15:
        break
print("Potentially ambiguous word occurrences:")
for item in ambiguous_examples[:15]:
    print(item)

In [ ]:
# Cell 6 — Q1(c): Build word and UPOS mappings from TRAINING DATA ONLY
PAD_WORD, UNK_WORD = "<PAD>", "<UNK>"
word_counter = collections.Counter(w.lower() for sent, _ in train_data for w in sent)

word2id = {PAD_WORD: 0, UNK_WORD: 1}
for word, count in word_counter.items():
    if word not in word2id:
        word2id[word] = len(word2id)

tag_set = sorted({tag for _, tags in train_data for tag in tags})
tag2id = {tag: i for i, tag in enumerate(tag_set)}
id2tag = {i: tag for tag, i in tag2id.items()}

print("Vocabulary size (including PAD/UNK):", len(word2id))
print("UPOS label count:", len(tag2id))
print("UPOS labels:", tag_set)
print("Word mapping sample:", list(word2id.items())[:15])
print("Tag mapping:", tag2id)

In [ ]:
# Cell 7 — Q1(d,e): Encode sequences, pad a mini-batch, create valid-token mask
def encode_data(data):
    encoded = []
    for words, tags in data:
        x = [word2id.get(w.lower(), word2id[UNK_WORD]) for w in words]
        y = [tag2id[t] for t in tags]
        encoded.append((words, tags, x, y))
    return encoded

train_encoded = encode_data(train_data)
dev_encoded   = encode_data(dev_data)
test_encoded  = encode_data(test_data)

def pad_batch(examples):
    max_len = max(len(x[2]) for x in examples)
    X = torch.full((len(examples), max_len), word2id[PAD_WORD], dtype=torch.long)
    Y = torch.full((len(examples), max_len), -100, dtype=torch.long)
    mask = torch.zeros((len(examples), max_len), dtype=torch.bool)
    for i, (_, _, x, y) in enumerate(examples):
        X[i, :len(x)] = torch.tensor(x)
        Y[i, :len(y)] = torch.tensor(y)
        mask[i, :len(x)] = True
    return X, Y, mask

mini = train_encoded[:4]
X_demo, Y_demo, mask_demo = pad_batch(mini)
lengths = [len(x) for _, _, x, _ in train_encoded]
print("Mini-batch token IDs shape:", tuple(X_demo.shape))
print("Mini-batch label IDs shape:", tuple(Y_demo.shape))
print("Valid-token mask shape:", tuple(mask_demo.shape))
print("Token IDs:\\n", X_demo)
print("Label IDs (-100 means padding):\\n", Y_demo)
print("Mask:\\n", mask_demo)
print("Sequence length min/mean/max:", min(lengths), round(float(np.mean(lengths)), 2), max(lengths))
print("Vocabulary size:", len(word2id), "| UPOS labels:", len(tag2id))

## Q2 — Bidirectional LSTM model

In [ ]:
# Cell 8 — Dataset, collate function, and BiLSTM architecture
class POSDataset(Dataset):
    def __init__(self, encoded): self.data = encoded
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

def collate_bilstm(batch):
    X, Y, mask = pad_batch(batch)
    return X, Y, mask

BATCH_SIZE = 32
EMBED_DIM = 100
HIDDEN_DIM = 128
EPOCHS_BILSTM = 8
LEARNING_RATE = 1e-3

train_loader = DataLoader(POSDataset(train_encoded), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_bilstm)
dev_loader = DataLoader(POSDataset(dev_encoded), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_bilstm)
test_loader = DataLoader(POSDataset(test_encoded), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_bilstm)

class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, num_tags, emb_dim=100, hidden_dim=128, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.25)
        self.classifier = nn.Linear(hidden_dim * 2, num_tags)
    def forward(self, input_ids):
        emb = self.embedding(input_ids)                  # [batch, seq, emb_dim]
        lstm_out, _ = self.lstm(emb)                      # [batch, seq, 2*hidden_dim]
        logits = self.classifier(self.dropout(lstm_out))  # [batch, seq, num_tags]
        return logits

bilstm = BiLSTMTagger(len(word2id), len(tag2id), EMBED_DIM, HIDDEN_DIM).to(device)
print(bilstm)
sample_X, sample_Y, sample_mask = next(iter(train_loader))
with torch.no_grad():
    emb_shape = bilstm.embedding(sample_X.to(device)).shape
    lstm_shape = bilstm.lstm(bilstm.embedding(sample_X.to(device)))[0].shape
    logits_shape = bilstm(sample_X.to(device)).shape
print("Input:", tuple(sample_X.shape))
print("Embedding output:", tuple(emb_shape))
print("BiLSTM output:", tuple(lstm_shape))
print("Tag-score logits:", tuple(logits_shape))

In [ ]:
# Cell 9 — Q2(b,d): Train with masked cross-entropy and early stopping
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(bilstm.parameters(), lr=LEARNING_RATE)

def run_bilstm_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, batches = 0.0, 0
    for X, Y, mask in loader:
        X, Y = X.to(device), Y.to(device)
        if training: optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = model(X)
            loss = criterion(logits.reshape(-1, len(tag2id)), Y.reshape(-1))
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += loss.item(); batches += 1
    return total_loss / max(batches, 1)

history = {"train": [], "validation": []}
best_dev = float("inf")
best_state = None
patience, wait = 2, 0
start_time = time.time()

for epoch in range(1, EPOCHS_BILSTM + 1):
    tr_loss = run_bilstm_epoch(bilstm, train_loader, optimizer)
    dv_loss = run_bilstm_epoch(bilstm, dev_loader)
    history["train"].append(tr_loss); history["validation"].append(dv_loss)
    print(f"Epoch {epoch}: train_loss={tr_loss:.4f}, val_loss={dv_loss:.4f}")
    if dv_loss < best_dev:
        best_dev = dv_loss
        best_state = copy.deepcopy(bilstm.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping.")
            break

if best_state is not None:
    bilstm.load_state_dict(best_state)
bilstm_train_seconds = time.time() - start_time

plt.figure(figsize=(7,4))
plt.plot(history["train"], label="Training loss")
plt.plot(history["validation"], label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss"); plt.title("BiLSTM training/validation loss")
plt.legend(); plt.grid(True); plt.show()
print(f"BiLSTM training time: {bilstm_train_seconds:.1f} seconds")

In [ ]:
# Cell 10 — Q2(e): BiLSTM test metrics and prediction helper
def bilstm_predict_dataset(model, encoded):
    model.eval()
    y_true, y_pred = [], []
    predictions = []
    with torch.no_grad():
        for start in range(0, len(encoded), BATCH_SIZE):
            batch = encoded[start:start+BATCH_SIZE]
            X, Y, mask = pad_batch(batch)
            logits = model(X.to(device))
            pred = logits.argmax(-1).cpu().numpy()
            for i, item in enumerate(batch):
                n = len(item[3])
                gold = item[3]
                guesses = pred[i, :n].tolist()
                y_true.extend(gold); y_pred.extend(guesses)
                predictions.append(guesses)
    return y_true, y_pred, predictions

bilstm_true, bilstm_pred, bilstm_test_predictions = bilstm_predict_dataset(bilstm, test_encoded)

def metric_dict(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(tag2id))), average="macro", zero_division=0
    )
    return {"accuracy": accuracy_score(y_true, y_pred), "macro_precision": p,
            "macro_recall": r, "macro_f1": f1}

bilstm_metrics = metric_dict(bilstm_true, bilstm_pred)
print("BiLSTM test metrics:", bilstm_metrics)
print(classification_report(bilstm_true, bilstm_pred,
      labels=list(range(len(tag2id))), target_names=[id2tag[i] for i in range(len(tag2id))],
      zero_division=0))

## Q3 — Fine-tune BERT with first-subword alignment

In [ ]:
# Cell 11 — BERT tokenizer, word-to-subword demonstration, and aligned dataset
MODEL_NAME = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

demo_words = ["The", "unbelievable", "runner", "won", "."]
demo = tokenizer(demo_words, is_split_into_words=True, return_tensors="pt")
print("Original words:", demo_words)
print("Subword tokens:", tokenizer.convert_ids_to_tokens(demo["input_ids"][0]))
print("Word IDs:", demo.word_ids(batch_index=0))

class BERTPOSDataset(Dataset):
    def __init__(self, data):
        self.items = []
        for words, tags in data:
            enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=256)
            word_ids = enc.word_ids()
            aligned = []
            previous_word = None
            for word_id in word_ids:
                if word_id is None:
                    aligned.append(-100)  # [CLS], [SEP], etc.
                elif word_id != previous_word:
                    aligned.append(tag2id[tags[word_id]])  # first subword gets the word label
                else:
                    aligned.append(-100)  # later subwords ignored
                previous_word = word_id
            enc["labels"] = aligned
            self.items.append(enc)
    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

def bert_collate(batch):
    features = [{"input_ids": x["input_ids"], "attention_mask": x["attention_mask"]} for x in batch]
    padded = tokenizer.pad(features, padding=True, return_tensors="pt")
    max_len = padded["input_ids"].shape[1]
    labels = [x["labels"] + [-100] * (max_len - len(x["labels"])) for x in batch]
    padded["labels"] = torch.tensor(labels, dtype=torch.long)
    return padded

bert_train_ds = BERTPOSDataset(train_data)
bert_dev_ds = BERTPOSDataset(dev_data)
bert_test_ds = BERTPOSDataset(test_data)

BERT_BATCH_SIZE = 16
bert_train_loader = DataLoader(bert_train_ds, batch_size=BERT_BATCH_SIZE, shuffle=True, collate_fn=bert_collate)
bert_dev_loader = DataLoader(bert_dev_ds, batch_size=BERT_BATCH_SIZE, shuffle=False, collate_fn=bert_collate)
bert_test_loader = DataLoader(bert_test_ds, batch_size=BERT_BATCH_SIZE, shuffle=False, collate_fn=bert_collate)

print("BERT train/dev/test sentences:", len(bert_train_ds), len(bert_dev_ds), len(bert_test_ds))
b = next(iter(bert_train_loader))
print({k: tuple(v.shape) for k, v in b.items()})

In [ ]:
# Cell 12 — Q3(c): Fine-tune pretrained BERT token classifier
bert_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(tag2id), id2label=id2tag,
    label2id=tag2id
).to(device)

BERT_EPOCHS = 3
BERT_LR = 2e-5
bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=BERT_LR)
total_steps = len(bert_train_loader) * BERT_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    bert_optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

def run_bert_epoch(model, loader, training=False):
    model.train(training)
    total_loss, n_batches = 0.0, 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        if training: bert_optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            out = model(**batch)
            loss = out.loss
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                bert_optimizer.step()
                scheduler.step()
        total_loss += loss.item(); n_batches += 1
    return total_loss / max(n_batches, 1)

bert_history = {"train": [], "validation": []}
best_bert_dev = float("inf")
best_bert_state = None
bert_patience, bert_wait = 1, 0
bert_start = time.time()

for epoch in range(1, BERT_EPOCHS + 1):
    tr = run_bert_epoch(bert_model, bert_train_loader, training=True)
    dv = run_bert_epoch(bert_model, bert_dev_loader, training=False)
    bert_history["train"].append(tr); bert_history["validation"].append(dv)
    print(f"BERT epoch {epoch}: train_loss={tr:.4f}, val_loss={dv:.4f}")
    if dv < best_bert_dev:
        best_bert_dev = dv
        best_bert_state = copy.deepcopy(bert_model.state_dict())
        bert_wait = 0
    else:
        bert_wait += 1
        if bert_wait >= bert_patience:
            print("Early stopping.")
            break

if best_bert_state is not None:
    bert_model.load_state_dict(best_bert_state)
bert_train_seconds = time.time() - bert_start

plt.figure(figsize=(7,4))
plt.plot(bert_history["train"], label="Training loss")
plt.plot(bert_history["validation"], label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("BERT training/validation loss")
plt.legend(); plt.grid(True); plt.show()
print(f"BERT fine-tuning time: {bert_train_seconds:.1f} seconds")

In [ ]:
# Cell 13 — Q3(d,e): Evaluate BERT on the held-out test set and compare
def evaluate_bert(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            pred = logits.argmax(-1)
            valid = batch["labels"] != -100
            y_true.extend(batch["labels"][valid].cpu().tolist())
            y_pred.extend(pred[valid].cpu().tolist())
    return y_true, y_pred

bert_true, bert_pred = evaluate_bert(bert_model, bert_test_loader)
bert_metrics = metric_dict(bert_true, bert_pred)
print("BERT test metrics:", bert_metrics)

comparison = pd.DataFrame([
    {"Model": "BiLSTM", **bilstm_metrics, "training_time_seconds": bilstm_train_seconds},
    {"Model": "BERT", **bert_metrics, "training_time_seconds": bert_train_seconds}
])
display(comparison.round(4))

## Q4 — Confusion matrices, errors, ambiguity, and discussion

In [ ]:
# Cell 14 — Q4(a): Confusion matrices (padding and ignored labels already excluded)
labels_order = list(range(len(tag2id)))
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, yt, yp, title in [
    (axes[0], bilstm_true, bilstm_pred, "BiLSTM confusion matrix"),
    (axes[1], bert_true, bert_pred, "BERT confusion matrix")
]:
    cm = confusion_matrix(yt, yp, labels=labels_order)
    sns.heatmap(cm, cmap="Blues", xticklabels=tag_set, yticklabels=tag_set, ax=ax)
    ax.set_title(title); ax.set_xlabel("Predicted UPOS"); ax.set_ylabel("Actual UPOS")
    ax.tick_params(axis="x", rotation=90); ax.tick_params(axis="y", rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 15 — Q4(b): At least five incorrect predictions per model
def collect_bilstm_errors(encoded, predictions, limit=10):
    errors = []
    for (words, tags, _, gold_ids), pred_ids in zip(encoded, predictions):
        for i, (g, p) in enumerate(zip(gold_ids, pred_ids)):
            if g != p:
                errors.append({"Sentence": " ".join(words), "Target word": words[i],
                               "Actual tag": id2tag[g], "Predicted tag": id2tag[p]})
                if len(errors) >= limit: return pd.DataFrame(errors)
    return pd.DataFrame(errors)

def collect_bert_errors(data, model, limit=10):
    errors = []
    model.eval()
    with torch.no_grad():
        for words, tags in data:
            enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=256, return_tensors="pt")
            word_ids = tokenizer(words, is_split_into_words=True, truncation=True, max_length=256).word_ids()
            logits = model(input_ids=enc["input_ids"].to(device),
                           attention_mask=enc["attention_mask"].to(device)).logits[0]
            pred_ids = logits.argmax(-1).cpu().tolist()
            seen = set()
            for token_pos, word_id in enumerate(word_ids):
                if word_id is None or word_id in seen: continue
                seen.add(word_id)
                if word_id >= len(tags): continue
                pred_tag = id2tag[pred_ids[token_pos]]
                if pred_tag != tags[word_id]:
                    errors.append({"Sentence": " ".join(words), "Target word": words[word_id],
                                   "Actual tag": tags[word_id], "Predicted tag": pred_tag})
                    if len(errors) >= limit: return pd.DataFrame(errors)
    return pd.DataFrame(errors)

bilstm_errors = collect_bilstm_errors(test_encoded, bilstm_test_predictions, limit=10)
bert_errors = collect_bert_errors(test_data, bert_model, limit=10)
print("BiLSTM incorrect predictions:")
display(bilstm_errors.head(5))
print("BERT incorrect predictions:")
display(bert_errors.head(5))

In [ ]:
# Cell 16 — Q4(c): Inspect ambiguous forms such as 'book' in different contexts
def find_word_contexts(data, word_to_find="book", max_examples=20):
    found = []
    for words, tags in data:
        for i, (w, t) in enumerate(zip(words, tags)):
            if w.lower() == word_to_find.lower():
                found.append({"Sentence": " ".join(words), "Word": w, "UPOS": t})
                if len(found) >= max_examples: return pd.DataFrame(found)
    return pd.DataFrame(found)

book_examples = find_word_contexts(train_data, "book")
display(book_examples)
print("If this corpus contains both NOUN and VERB uses, compare the surrounding grammatical context.")
print("If not, repeat with another form from the ambiguity list (e.g. like, work, play, watch).")

In [ ]:
# Cell 17 — Q4(d,e): Error analysis by sentence length and OOV/rare words
train_word_freq = collections.Counter(w.lower() for words, _ in train_data for w in words)
test_records = []
for idx, ((words, tags, _, gold_ids), pred_ids) in enumerate(zip(test_encoded, bilstm_test_predictions)):
    wrong = sum(g != p for g, p in zip(gold_ids, pred_ids))
    oov_count = sum(w.lower() not in word2id for w in words)
    rare_count = sum(train_word_freq[w.lower()] <= 2 for w in words)
    test_records.append({"sentence_id": idx, "length": len(words), "errors": wrong,
                         "error_rate": wrong / len(words), "oov_tokens": oov_count,
                         "rare_tokens": rare_count})
error_analysis = pd.DataFrame(test_records)
display(error_analysis.sort_values("error_rate", ascending=False).head(10))
print("Average BiLSTM error rate by sentence-length group:")
error_analysis["length_group"] = pd.cut(error_analysis["length"], bins=[0,10,20,40,10000],
                                        labels=["1–10","11–20","21–40","41+"])
display(error_analysis.groupby("length_group", observed=False)[["error_rate","oov_tokens","rare_tokens"]].mean())

print("Discussion prompts (write using your measured results):")
print("1. Which model has higher test accuracy and macro-F1? Quote the comparison table.")
print("2. Does BERT handle context-dependent words better in the examples you inspected?")
print("3. Does BiLSTM make more errors on long sentences, rare words, or OOV words?")
print("4. BiLSTM is usually lighter/faster; BERT has contextual pretrained representations but higher compute cost.")
print("5. Possible improvements: tune hyperparameters on dev only, use pretrained word vectors for BiLSTM,")
print("   increase training epochs with early stopping, handle long sequences carefully, and inspect annotation ambiguity.")